# Tomato sc sequencing analysis: 01 preprocessing

Loads raw 10x Genomics CellRanger output for tomato leaf single-cell RNA-seq (healthy vs. tomato chlorosis virus-infected), performs quality control (mitochondrial/chloroplast content, count-based outlier filtering), doublet detection, and normalization. 
This produces the processed, normalized AnnData object used for clustering and cell-type annotation in the next notebook.

**Input**: `dataset/raw/` (barcodes.tsv, features.tsv, matrix.mtx — CellRanger output),
`dataset/reference/ITAG4.0_goterms.txt` (GO annotation for QC gene sets)

**Output**: `dataset/processed/tomato_qc_normalized.h5ad`

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad

import re
from collections import Counter

from scipy.stats import median_abs_deviation

## 1. Save data as AnnData object

In [2]:
DATA_PATH = '../dataset/raw/'

# Inspect raw CellRanger file formats before parsing, to confirm the expected 10x mtx structure  

# 1. barcodes.tsv (cell IDs)
print("--- barcodes.tsv ---")
with open(f'{DATA_PATH}barcodes.tsv', 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

# 2. features.tsv (gene IDs)
print("\n--- features.tsv ---")
with open(f'{DATA_PATH}features.tsv', 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

# 3. matrix.mtx (sparse count matrix in MatrixMarket format)
print("\n--- matrix.mtx ---")
with open(f'{DATA_PATH}matrix.mtx', 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

--- barcodes.tsv ---
AAACCCAAGCATCTTG-1
AAACCCAAGGCGACAT-1
AAACCCAAGGGTATAT-1
AAACCCACAACTCCAA-1
AAACCCACAATTCACG-1

--- features.tsv ---
gene:Solyc00g500001.1	Solyc00g500001.1	Gene Expression
gene:Solyc00g500002.1	Solyc00g500002.1	Gene Expression
gene:Solyc00g500003.1	Solyc00g500003.1	Gene Expression
gene:Solyc00g500004.1	Solyc00g500004.1	Gene Expression
gene:Solyc00g500005.1	Solyc00g500005.1	Gene Expression

--- matrix.mtx ---
%%MatrixMarket matrix coordinate integer general
%metadata_json: {"software_version": "cellranger-5.0.0", "format_version": 2}
34075 44867 46330391
24 1 3
30 1 3


In [3]:
data_path = '../dataset/raw/'

adata = sc.read_10x_mtx(data_path)

In [4]:
adata

AnnData object with n_obs × n_vars = 44867 × 34075
    var: 'gene_ids', 'feature_types'

In [5]:
adata.var.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34075 entries, Solyc00g500001.1 to Solyc12g100367.1
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   gene_ids       34075 non-null  object
 1   feature_types  34075 non-null  object
dtypes: object(2)
memory usage: 1.8+ MB


In [6]:
# Barcode suffix (-1, -2, ...) identifies which sample each cell came from;
# needed to assign healthy/infected labels below
adata.obs['sample_no'] = adata.obs.index.str.split('-').str[-1]
print(adata.obs['sample_no'].value_counts())

sample_no
1    13720
4     9506
5     8641
3     7000
2     6000
Name: count, dtype: int64


The source publication reports 13,720 healthy cells and 9,506 infected cells.

These counts match samples 1 and 4 almost exactly, so they are assigned as the healthy and infected reference samples;
the remaining samples are treated as additional infected replicates.

In [7]:
healthy = 1
infected = 4

## 2. Quality control

### 2-1) Reference annotation(data schema verification)

In [8]:
feature_path = '../dataset/raw/features.tsv'

feature_df = pd.read_csv(feature_path, sep='\t')

In [9]:
feature_df.head(20)

,gene:Solyc00g500001.1,Solyc00g500001.1,Gene Expression
0,gene:Solyc00g500002.1,Solyc00g500002.1,Gene Expression
1,gene:Solyc00g500003.1,Solyc00g500003.1,Gene Expression
2,gene:Solyc00g500004.1,Solyc00g500004.1,Gene Expression
3,gene:Solyc00g500005.1,Solyc00g500005.1,Gene Expression
4,gene:Solyc00g500006.1,Solyc00g500006.1,Gene Expression
5,gene:Solyc00g500007.1,Solyc00g500007.1,Gene Expression
6,gene:Solyc00g500008.1,Solyc00g500008.1,Gene Expression
7,gene:Solyc00g500009.1,Solyc00g500009.1,Gene Expression
8,gene:Solyc00g500010.1,Solyc00g500010.1,Gene Expression
9,gene:Solyc00g500011.1,Solyc00g500011.1,Gene Expression


In [10]:
features = pd.read_csv("../dataset/raw/features.tsv", sep="\t", header=None,
                        names=["gene_id", "gene_symbol", "feature_type"])

# Confirm no other assay type (e.g. ATAC, antibody capture) is mixed into this expression matrix before proceeding
print(features["feature_type"].value_counts())
assert features["feature_type"].isin(["Gene Expression"]).all(), "Unexpected feature_type found"

feature_type
Gene Expression    34075
Name: count, dtype: int64


Gene IDs follow the ITAG4.0 tomato genome annotation. This dataset does not flag mitochondrial/chloroplast genes directly, so GO terms from the ITAG4.0 annotation are used to identify them for the QC metrics below.

In [11]:
# Load ITAG4.0 GO term annotation (gene_id -> list of GO terms)
go_path = "../dataset/reference/ITAG4.0_goterms.txt"

records = []
with open(go_path) as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        gene_id = parts[0]
        go_terms = parts[1].split(",") if len(parts) > 1 else []
        records.append((gene_id, go_terms))

go_df = pd.DataFrame(records, columns=["gene_id", "go_terms"])
print(go_df.shape)
print(go_df.head(10))

(33562, 2)
            gene_id                              go_terms
0  Solyc01g004000.1                                    []
1  Solyc01g004002.1                                    []
2  Solyc01g004004.1                                    []
3  Solyc01g004006.1                                    []
4  Solyc01g004008.1                                    []
5  Solyc01g005000.2  [GO:0016831, GO:0019752, GO:0030170]
6  Solyc01g005020.2                                    []
7  Solyc01g005040.2                                    []
8  Solyc01g005060.2                                    []
9  Solyc01g005070.2                          [GO:0005543]


Use GO ID

In [12]:
# Strip version suffixes so gene IDs match consistently between the expression matrix and the GO annotation file
go_df["base_id"] = go_df["gene_id"].str.replace(r"\.\d+$", "", regex=True)
go_lookup = go_df.drop_duplicates("base_id").set_index("base_id")

bare_ids = adata.var_names.str.replace("^gene:", "", regex=True)
base_ids = bare_ids.str.replace(r"\.\d+$", "", regex=True)

mapped = go_lookup.reindex(base_ids)

# Mitochondrial and chloroplast GO terms, used as QC gene sets below (ribosomal term kept for reference, not used in filtering)
MT_GO = {"GO:0005739"}
CHLORO_GO = {"GO:0009507"}
RIBO_GO = {"GO:0005840"}

adata.var["mt"] = mapped["go_terms"].apply(lambda t: bool(MT_GO & set(t)) if isinstance(t, list) else False).values
adata.var["chloro"] = mapped["go_terms"].apply(lambda t: bool(CHLORO_GO & set(t)) if isinstance(t, list) else False).values

print(adata.var[["mt", "chloro"]].sum())

mt         8
chloro    49
dtype: int64


In [14]:
# Compute standard QC metrics (total_counts, n_genes_by_counts, pct_counts_mt, pct_counts_chloro, and their log1p versions)
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "chloro"], inplace=True, log1p=True)

In [15]:
adata.obs["pct_counts_mt"].describe()

count    44867.000000
mean         0.013002
std          0.032586
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.445434
Name: pct_counts_mt, dtype: float64

In [16]:
adata.obs["pct_counts_chloro"].describe()

count    44867.000000
mean         0.054031
std          0.138465
min          0.000000
25%          0.000000
50%          0.000000
75%          0.018188
max          2.363368
Name: pct_counts_chloro, dtype: float64

In [17]:
quantiles =  [0.95, 0.97, 0.99]

for q in quantiles:
    print(f'{q*100}% value of mt: {adata.obs["pct_counts_mt"].quantile(q)}')
print('\n')

for q in quantiles:
    print(f'{q*100}% value of chloro: {adata.obs["pct_counts_chloro"].quantile(q)}')

95.0% value of mt: 0.07910766452550883
97.0% value of mt: 0.10330791607499089
99.0% value of mt: 0.15847861766815186


95.0% value of chloro: 0.3435983568429943
97.0% value of chloro: 0.43720420539379085
99.0% value of chloro: 0.6556285715103144


In [18]:
adata.obs["total_counts"].describe()

count    44867.000000
mean      2565.718994
std       2726.122314
min        307.000000
25%       1021.000000
50%       1781.000000
75%       2808.000000
max      40995.000000
Name: total_counts, dtype: float64

In [19]:
adata.obs["n_genes_by_counts"].describe()

count    44867.000000
mean      1032.616199
std        614.518087
min         94.000000
25%        616.000000
50%        876.000000
75%       1245.000000
max       6800.000000
Name: n_genes_by_counts, dtype: float64

In [20]:
adata.obs["log1p_total_counts"].describe()

count    44867.000000
mean         7.502445
std          0.793163
min          5.730100
25%          6.929517
50%          7.485492
75%          7.940584
max         10.621230
Name: log1p_total_counts, dtype: float64

## Preprocessing criteria

- `n_genes_by_counts`, `total_counts` → log1p-transformed, then filtered by MAD (median absolute deviation)-based outlier detection
  (n=5, Scanpy's recommended default). Count data is right-skewed, so MAD is used instead of a normal-distribution-based cutoff.
- `pct_counts_mt`, `pct_counts_chloro` → percentile-based cutoff (top 1%). Elevated organellar RNA content typically indicates damaged or dying cells.

In [21]:
# Cells above the 99th percentile of mt/chloroplast content are flagged as outliers
preprocess_quantile = 0.99

mt_cutoff = adata.obs["pct_counts_mt"].quantile(preprocess_quantile)
chloro_cutoff = adata.obs["pct_counts_chloro"].quantile(preprocess_quantile)

outlier_mt = adata.obs["pct_counts_mt"] > mt_cutoff
outlier_chloro = adata.obs["pct_counts_chloro"] > chloro_cutoff

In [22]:
def mad_outlier_detect(data, mad_n=5):
    """
    Flag values more than `mad_n` median absolute deviations (MAD)
    from the median. Used instead of a z-score cutoff because count
    data does not follow a normal distribution.

    Args:
        data (pd.Series): values to test (log1p-transformed counts)
        mad_n (int): number of MADs defining the outlier threshold

    Returns:
        pd.Series[bool]: True where the value is an outlier
    """
    median = data.median()
    mad = median_abs_deviation(data)
    lower_bound = median - mad_n * mad
    upper_bound = median + mad_n * mad
    return (data < lower_bound) | (data > upper_bound)

outlier_total = mad_outlier_detect(adata.obs["log1p_total_counts"])
outlier_n_genes = mad_outlier_detect(adata.obs["log1p_n_genes_by_counts"])

In [23]:
# A cell is discarded if it fails any one of the four QC criteria
adata.obs['outlier'] = outlier_mt | outlier_chloro | outlier_total | outlier_n_genes
adata_raw = adata.copy()
adata = adata[~adata.obs["outlier"]].copy()

In [26]:
print(f"Before filtering: {adata_raw.n_obs}, After filtering: {adata.n_obs}")

필터링 전: 44867, 필터링 후: 43927


## 3. Doublet detection

A droplet can occasionally capture two or more cells, producing a
hybrid transcriptome with no clear biological identity. Scrublet
flags likely doublets by comparing each cell's expression profile to
synthetic doublets generated by summing pairs of real cells.

In [28]:
# Compute per-cell doublet scores and classify likely doublets
sc.pp.scrublet(adata)

adata.obs["predicted_doublet"]

AAACCCAAGCATCTTG-1    False
AAACCCAAGGCGACAT-1    False
AAACCCAAGGGTATAT-1    False
AAACCCACAACTCCAA-1    False
AAACCCACAATTCACG-1    False
                      ...  
TTTGTTGCATCGTGGC-5    False
TTTGTTGTCAAATGAG-5    False
TTTGTTGTCCATCTCG-5    False
TTTGTTGTCCTACCGT-5    False
TTTGTTGTCGCGCTGA-5    False
Name: predicted_doublet, Length: 43927, dtype: bool

In [32]:
# Check whether any cells were flagged as doublets
adata.obs["predicted_doublet"][adata.obs["predicted_doublet"] == False].count()

np.int64(43927)

In [29]:
adata.obs["doublet_score"].describe()

count    43927.000000
mean         0.042363
std          0.031206
min          0.003048
25%          0.021170
50%          0.034262
75%          0.054774
max          0.538194
Name: doublet_score, dtype: float64

## 4. Normalization

Sequencing depth varies between cells, so raw counts are not directly
comparable across cells. Library-size normalization followed by
log1p transformation puts all cells on a comparable scale for
downstream clustering and differential expression analysis.

In [41]:
# Preserve raw counts separately in case a downstream step
# (e.g. differential expression) requires unnormalized values
adata.layers["counts"] = adata.X.copy()

In [42]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

## 5. Save preprocessed data

In [44]:
adata.write_h5ad("../dataset/processed/tomato_qc_normalized.h5ad")